Can we do some EPE Expected Prediction Error and Generization error practice?

First we're gonna need a sample problem

We could try something simple like Gaussian distributions, but do X and Y need to rely on each other?

Even simpler to start would be a simple equation...

The idea is that for EPE we need just a best guess f

And for generalization we would need to generate the f by

We won't even be able to compute the integral itself, so we need a numerical method like monte carlo

There are 2 parts of the EPE formula within the integral. The PDF and L2 error between y and f

This is a joint PDF for values of x and y between 0 and 1

In [1]:
def p (x,y):
    return x + 1.5 * y ** 2

we're gonna need an f which is a best guess for predicting y based on x

Or even try to calculate it

our predictor f(x) = E(Y|X=x) = integral (y * p(y|x)) dy

So we're gonna need p(y|x) which equals p(x,y) / p(x)

p(x) is marginal density or integral( p(x,y) ) dy = x + .5

meaing our predictor is integral 0 to 1 ( y *  (x + 1.5 * y^2) / (x - .5) ) dy o 1/(x-.5) * int (yx + 1.5y^3)

giving us (4x + 3) / (4(2x+1))

ok but maybe we shouldn't actually use this optimal predictor for our testing purposes, but rather a "good guess"

In [2]:
def f_0 (x):
    return .5

In [3]:
def f_1 (x):
    return 1 - x**2

In [4]:
def f_2 (x):
    return 1 - .5 * x

In [23]:
def f_star(x):
    return (4*x + 3) / (4*(2*x + 1))

Here are 3 guesses, and we can try to compare EPE for them

In [11]:
import numpy as np

first generate our test data from the joint pdf

In [22]:
def sample_from_joint (n): # n is number of samples
    data = []
    max_density = 2.5

    while len(data) < n:
        x = np.random.uniform(0,1)
        y = np.random.uniform(0,1)
        rand = np.random.uniform(0,1)

        prob = p(x,y) / max_density

        if rand < prob:
            # accept sample, so add this to our samples
            data.append((x,y))
    return np.array(data)


calculate empirical error L2

In [24]:
n = 200
data = sample_from_joint(n)

print(np.mean((data[:,1] - f_0 (data[:,0]) )** 2))
print(np.mean((data[:,1] - f_1 (data[:,0]) )** 2))
print(np.mean((data[:,1] - f_2 (data[:,0]) )** 2))
print(np.mean((data[:,1] - f_star (data[:,0]) )** 2))

0.08261580339061605
0.14206633943043828
0.08516644681405165
0.06636212852118416


Now let's try Generalization error

This means we generate f by training on the dataset data, which is a sample generated by our probability dist

we could do this via linear regression?

We would just need the 2 formulas for B0 and B1 and then to train them on data

In [30]:
def B_1 (x, y):
    n = len(x)
    sum_x = np.sum(x)
    sum_y = np.sum(y)
    sum_xy = np.sum(x * y)
    sum_xx = np.sum(x * x)

    return ( n * sum_xy - sum_x * sum_y ) / (n*sum_xx - sum_x * sum_x)

In [31]:
def B_0 (B_1, x, y):
    n = len(x)
    sum_x = np.sum(x)
    sum_y = np.sum(y)

    return (sum_y - B_1 * sum_x) / n

In [32]:
def linear_reg (B_0, B_1, x):
    return B_0 + B_1 * x

In [33]:
x = data[:,0]
y = data[:,1]

B_1 = B_1(x, y)
B_0 = B_0(B_1, x, y)

In [34]:
print(np.mean((data[:,1] - linear_reg(B_0, B_1, data[:,0]))**2))

0.06596143625868742
